Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: 제미나이(Gemini ) API 호출 코드 내 429 할당량 초과 에러 방어를 위한 지수 백오프(Exponential Backoff ), 지터(Jitter ), 폴백 모델(Fallback Model ) 적용 여부를 정밀 진단하는 스크립트다.

## 제미나이 API 호출 복원력 자가 진단 도구 (Gemini API Resilience Checker )

### 1. 진단 초기화 및 전역 설정

장애 극복 회복탄력성 스캐너의 스캔 대상 경로를 정의하고 전역 상태 변수를 선언한다.

In [ ]:
import os
import re

SCAN_PATH = "."
FOUND_COUNT = 0

### 2. 복원력 패턴 정밀 자가 진단 실행

프로젝트 내 파일들을 분석하여 제미나이 API 호출부의 429 장애 극복 회복탄력성 설계 상태를 자가 점검한다.

**정밀 검증 대상 복원력 패턴:**
* **지수 백오프 및 재시도**: 일시적인 네트워크 순제한이나 429 에러 발생 시 순차적으로 지수 백오프 기반 재요청 패턴이 적용되어 있는지 검사한다.
* **지터 무작위 대기**: 재시도 시점이 동일하게 집중되는 병목 현상을 방지하기 위해 랜덤한 대기 시간(지터 )을 섞어 주었는지 검사한다.
* **최대 재시도 제한**: 영구 장애 시 무한 루프 과금을 방지하는 최대 재시도 제약 장치가 존재하는지 검증한다.
* **폴백 모델 예외 복구**: 429 할당량 소진 또는 Pro 모델 호출 실패 시 차선책인 가벼운 Flash 모델 등으로 예외를 복구하는 Fallback 모델 전환 구조를 권장 분석한다.

In [ ]:
if not os.path.isdir(SCAN_PATH):
  print(f"[오류] 지정한 경로가 디렉터리가 아니거나 존재하지 않는다: {SCAN_PATH}")
else:
  print(f"[검사 시작] 제미나이 API 호출 부위의 429 장애 극복 회복탄력성 패턴 자가 진단을 시작한다...")
  print(f"대상 디렉터리: {os.path.abspath(SCAN_PATH)}")
  print("-" * 72)
  
  for root, dirs, files in os.walk(SCAN_PATH):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    
    for file in files:
      if not (file.endswith(".py") or file.endswith(".sh")):
        continue
        
      file_path = os.path.join(root, file)
      
      if "gemini-api-resilience-checker" in file:
        continue
        
      try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
          content = f.read()
          
        has_gemini_call = bool(re.search(r"generate_content|generate_content_stream|generate_content_async", content))
        
        if has_gemini_call:
          print(f"[진단 대상 발견] 파일: {file_path}")
          
          has_retry = bool(re.search(r"retry|tenacity|backoff|sleep", content, re.IGNORECASE))
          has_jitter = bool(re.search(r"jitter|random|wait_random", content, re.IGNORECASE))
          has_max_retries = bool(re.search(r"stop_after|max_retries|stop=", content, re.IGNORECASE))
          has_fallback = bool(re.search(r"fallback|try:|except", content, re.IGNORECASE))
          
          if not has_retry:
            print("  [경고] 지수 백오프 및 재시도(Exponential Backoff & Retry ) 누락 의심!")
            print("           - 백오프나 재시도 제어 로직이 검출되지 않았다.")
            FOUND_COUNT += 1
          else:
            print("  [통과] 지수 백오프 및 재시도 제어 패턴 정상 감지")
            
          if not has_jitter:
            print("  [경고] 지터(Jitter ) 무작위 대기 기법 누락 의심!")
            print("           - 요청 폭주를 예방하는 지터 대기 로직이 검출되지 않았다.")
            FOUND_COUNT += 1
          else:
            print("  [통과] 지터 무작위 대기 기법 정상 감지")
            
          if not has_max_retries:
            print("  [경고] 최대 재시도 횟수(Max Retries ) 제한 설정 누락 의심!")
            print("           - 무한 재시도로 인한 비용 폭증을 방지하는 재시도 제한 장치가 검출되지 않았다.")
            FOUND_COUNT += 1
          else:
            print("  [통과] 최대 재시도 횟수 제한 설정 정상 감지")
            
          if not has_fallback:
            print("  [권장] 폴백 모델(Fallback Model ) 예외 복구 패턴 적용 검토")
            print("           - 가용성 유지를 위한 폴백 체인이 검출되지 않았다.")
            FOUND_COUNT += 1
          else:
            print("  [통과] 폴백 모델 예외 복구 패턴 정상 감지")
            
          print("-" * 72)
      except Exception as e:
        pass

  if FOUND_COUNT == 0:
    print("[성공] 프로젝트의 모든 제미나이 API 호출부가 429 장애 극복 회복탄력성 패턴을 안전하게 구비하고 있다!")
  else:
    print(f"[완료] 총 {FOUND_COUNT}개의 복원력 패턴 미흡 및 개선 권장 사항이 발견되었다.")
    print("     (참고: 할당량 한도 및 가용 잔여량은 GCP IAM 할당량 제어 콘솔 주소(https://console.cloud.google.com/iam-admin/quotas )를 통해 실시간으로 확인 가능하다.)")

### 3. 추천 표준 처방 가이드 - 파이썬(Python ) tenacity 데코레이터 표준 템플릿

아래는 `tenacity` 라이브러리를 활용해 지수 백오프, 지터, 최대 재시도를 결합하고 주 모델 실패 시 폴백 모델로 우회 처리하는 모범 코드 스키마다.

In [ ]:
from google import genai
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_random_exponential

@retry(
  wait=wait_random_exponential(min=1, max=60),
  stop=stop_after_attempt(5)
)
def generate_with_retry(client, model_id, prompt):
  return client.models.generate_content(
    model=model_id,
    contents=prompt
  )

def generate_with_fallback(client, prompt):
  try:
    return generate_with_retry(client, 'gemini-3.5-pro', prompt)
  except Exception as e:
    print(f'[안내] 주 모델 호출 실패로 폴백 모델 호출로 즉시 우회한다: {e}')
    return generate_with_retry(client, 'gemini-3.5-flash', prompt)